In [2]:
import os
import shutil
import tensorflow as tf

# ==========================================
# KONFIGURASI FOLDER (SILAKAN DISESUAIKAN)
# ==========================================
FOLDER_SUMBER = "data_cropped" 
FOLDER_TUJUAN = "data_balanced" 
NAMA_FOLDER_KELAS_4 = "4" # Sesuaikan dengan nama folder Kelas 4 Anda

# 1. Definisikan Pipeline Augmentasi (Sama persis dengan kode 65.05% Anda)
# Kita pastikan fill_mode='nearest' agar tidak ada segitiga hitam saat rotasi
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2, fill_mode='nearest'),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1, fill_mode='nearest')
])

def run_moderate_oversample(source_dir, output_dir, target_class):
    print(f"🚀 Memulai Moderate Oversampling dari '{source_dir}' ke '{output_dir}'...\n")
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    class_folders = [f for f in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, f))]
    total_asli = 0
    total_baru = 0

    for cls in class_folders:
        src_class_path = os.path.join(source_dir, cls)
        out_class_path = os.path.join(output_dir, cls)

        if not os.path.exists(out_class_path):
            os.makedirs(out_class_path)

        images = [img for img in os.listdir(src_class_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        count_asli = len(images)
        total_asli += count_asli
        
        print(f"-> Memproses Kelas '{cls}' ({count_asli} gambar asli)")

        for img_name in images:
            img_path = os.path.join(src_class_path, img_name)
            out_path_original = os.path.join(out_class_path, img_name)

            # Langkah A: COPY gambar aslinya (Jangan diubah!)
            shutil.copy2(img_path, out_path_original)

            # Langkah B: Lakukan Augmentasi HANYA untuk Kelas 4
            if cls == target_class:
                # Muat gambar menjadi array TensorFlow
                img = tf.keras.utils.load_img(img_path)
                img_array = tf.keras.utils.img_to_array(img)
                
                # Tambahkan dimensi batch (karena layer augmentasi butuh format batch)
                img_array = tf.expand_dims(img_array, 0)
                
                # Terapkan augmentasi (Wajib training=True agar augmentasinya aktif berputar/bergeser)
                augmented_img_tensor = data_augmentation(img_array, training=True)
                
                # Hapus dimensi batch dan simpan gambar baru
                augmented_img = augmented_img_tensor[0]
                
                # Buat nama file baru: "nama_asli_aug.png"
                name, ext = os.path.splitext(img_name)
                out_path_aug = os.path.join(out_class_path, f"{name}_aug{ext}")
                
                tf.keras.utils.save_img(out_path_aug, augmented_img)
        
        count_akhir = len(os.listdir(out_class_path))
        total_baru += count_akhir
        print(f"   Selesai! Jumlah di folder '{cls}' menjadi: {count_akhir} gambar.\n")

    print("=" * 60)
    print("✅ PROSES OVERSAMPLING SELESAI!")
    print(f"Total data SEBELUM: {total_asli}")
    print(f"Total data SESUDAH: {total_baru} (Kelas 4 berhasil digandakan!)")
    print(f"Gunakan folder '{output_dir}' untuk training K-Fold Anda.")
    print("=" * 60)

# Jalankan script
run_moderate_oversample(FOLDER_SUMBER, FOLDER_TUJUAN, NAMA_FOLDER_KELAS_4)

🚀 Memulai Moderate Oversampling dari 'data_cropped' ke 'data_balanced'...

-> Memproses Kelas '1' (195 gambar asli)
   Selesai! Jumlah di folder '1' menjadi: 195 gambar.

-> Memproses Kelas '2' (379 gambar asli)
   Selesai! Jumlah di folder '2' menjadi: 379 gambar.

-> Memproses Kelas '3' (239 gambar asli)
   Selesai! Jumlah di folder '3' menjadi: 239 gambar.

-> Memproses Kelas '4' (91 gambar asli)
   Selesai! Jumlah di folder '4' menjadi: 182 gambar.

✅ PROSES OVERSAMPLING SELESAI!
Total data SEBELUM: 904
Total data SESUDAH: 995 (Kelas 4 berhasil digandakan!)
Gunakan folder 'data_balanced' untuk training K-Fold Anda.


In [ ]:
import os
import shutil
import random
import tensorflow as tf

# ==========================================
# KONFIGURASI FOLDER & ATURAN
# ==========================================
FOLDER_SUMBER = "data_cropped" 
FOLDER_TUJUAN = "data_balanced_v2" # Gunakan folder baru agar tidak menimpa yang lama

# Aturan Oversampling (Dictionary)
# Format -> "nama_folder": persentase_data_yang_diaugmentasi
ATURAN_OVERSAMPLING = {
    "4": 1.0,  # Kelas 4: Ambil 100% datanya untuk diaugmentasi (91 -> 182)
    "3": 0.25   # Kelas 3: Ambil 50% datanya secara acak untuk diaugmentasi (239 -> ~358)
}

# Pipeline Augmentasi (Resep Pemenang Anda)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2, fill_mode='nearest'),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1, fill_mode='nearest')
])

def run_smart_oversample(source_dir, output_dir, rules):
    print(f"🚀 Memulai Smart Oversampling ke '{output_dir}'...\n")
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    class_folders = [f for f in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, f))]
    total_asli = 0
    total_baru = 0

    for cls in class_folders:
        src_class_path = os.path.join(source_dir, cls)
        out_class_path = os.path.join(output_dir, cls)

        if not os.path.exists(out_class_path):
            os.makedirs(out_class_path)

        images = [img for img in os.listdir(src_class_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        count_asli = len(images)
        total_asli += count_asli
        
        print(f"-> Memproses Kelas '{cls}' ({count_asli} gambar asli)")

        # Langkah 1: Selalu COPY semua gambar asli
        for img_name in images:
            img_path = os.path.join(src_class_path, img_name)
            out_path_original = os.path.join(out_class_path, img_name)
            shutil.copy2(img_path, out_path_original)

        # Langkah 2: Cek apakah kelas ini masuk daftar yang harus dioversample
        if cls in rules:
            ratio = rules[cls]
            num_to_augment = int(count_asli * ratio)
            
            print(f"   [!] Menerapkan oversampling sebesar {ratio*100}%...")
            print(f"   [!] Memilih {num_to_augment} gambar untuk diaugmentasi.")
            
            # Pilih gambar secara acak untuk diaugmentasi
            images_to_augment = random.sample(images, num_to_augment)
            
            for img_name in images_to_augment:
                img_path = os.path.join(src_class_path, img_name)
                
                # Proses Augmentasi
                img = tf.keras.utils.load_img(img_path)
                img_array = tf.keras.utils.img_to_array(img)
                img_array = tf.expand_dims(img_array, 0)
                
                augmented_img_tensor = data_augmentation(img_array, training=True)
                augmented_img = augmented_img_tensor[0]
                
                # Simpan dengan nama baru
                name, ext = os.path.splitext(img_name)
                out_path_aug = os.path.join(out_class_path, f"{name}_aug{ext}")
                tf.keras.utils.save_img(out_path_aug, augmented_img)
        
        count_akhir = len(os.listdir(out_class_path))
        total_baru += count_akhir
        print(f"   Selesai! Jumlah akhir di folder '{cls}': {count_akhir} gambar.\n")

    print("=" * 60)
    print("✅ PROSES SMART OVERSAMPLING SELESAI!")
    print(f"Total data SEBELUM: {total_asli}")
    print(f"Total data SESUDAH: {total_baru}")
    print(f"Gunakan folder '{output_dir}' untuk eksperimen K-Fold Anda selanjutnya.")
    print("=" * 60)

# Jalankan script
run_smart_oversample(FOLDER_SUMBER, FOLDER_TUJUAN, ATURAN_OVERSAMPLING)

🚀 Memulai Smart Oversampling ke 'data_balanced_v2'...

-> Memproses Kelas '1' (195 gambar asli)
   Selesai! Jumlah akhir di folder '1': 195 gambar.

-> Memproses Kelas '2' (379 gambar asli)
   Selesai! Jumlah akhir di folder '2': 379 gambar.

-> Memproses Kelas '3' (239 gambar asli)
   [!] Menerapkan oversampling sebesar 50.0%...
   [!] Memilih 119 gambar untuk diaugmentasi.
   Selesai! Jumlah akhir di folder '3': 358 gambar.

-> Memproses Kelas '4' (91 gambar asli)
   [!] Menerapkan oversampling sebesar 100.0%...
   [!] Memilih 91 gambar untuk diaugmentasi.
   Selesai! Jumlah akhir di folder '4': 182 gambar.

✅ PROSES SMART OVERSAMPLING SELESAI!
Total data SEBELUM: 904
Total data SESUDAH: 1114
Gunakan folder 'data_balanced_v2' untuk eksperimen K-Fold Anda selanjutnya.
